# **Word Embeddings**

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [8]:
#The indices of the words represent the values in the lookup table
word_to_idx = {"I": 0, "love": 1, "eating":2, "and":3, "sleeping":4}
word_index = torch.tensor([word_to_idx["love"]])
print(word_index)

tensor([1])


In [9]:
embeddings = nn.Embedding(5, 7)    # 5 words in vocab, embedding size is 7
# nn.Embedding initializes embeddings randomly and learns them during training

love = embeddings(word_index)
print(love)
print(love.shape)

tensor([[ 1.0377,  0.5544,  0.5584,  2.0119, -0.5391,  0.8138, -1.2161]],
       grad_fn=<EmbeddingBackward0>)
torch.Size([1, 7])


In [12]:
all_ind = torch.tensor([w for w in range(5)], dtype = torch.long)
all_words = embeddings(all_ind)

print(all_ind)
print(all_words)
print(all_words.shape)

tensor([0, 1, 2, 3, 4])
tensor([[ 0.1048,  0.1497, -0.9205,  2.3389,  1.2234, -0.9763,  0.2294],
        [ 1.0377,  0.5544,  0.5584,  2.0119, -0.5391,  0.8138, -1.2161],
        [-0.2734,  1.2781, -1.7792,  0.7978,  1.8049, -0.2368, -0.1693],
        [-1.4559, -1.2666, -0.6038,  1.3418, -0.4340,  0.1466,  3.9114],
        [-0.9305,  1.0644, -1.5127, -0.4928,  0.3101, -0.7125, -1.2661]],
       grad_fn=<EmbeddingBackward0>)
torch.Size([5, 7])


# **Exercise**

Given a sequence of words, we want to predict the ith word of the sequence: P(w(i)|w(i-1), w(i-2), .....)

In [13]:
CONTEXT_SIZE = 2
# At each training step, the model looks at 2 words on the left and 2 words on the right of the center word

EMBEDDING_DIM = 10
# Size of the embedding vector that represents each word

# SHAKESPEARE SONNET 2
test_sentence =  """When forty winters shall besiege thy brow,
                    And dig deep trenches in thy beauty's field,
                    Thy youth's proud livery so gazed on now,
                    Will be a totter'd weed of small worth held:
                    Then being asked, where all thy beauty lies,
                    Where all the treasure of thy lusty days;
                    To say, within thine own deep sunken eyes,
                    Were an all-eating shame, and thriftless praise.
                    How much more praise deserv'd thy beauty's use,
                    If thou couldst answer 'This fair child of mine
                    Shall sum my count, and make my old excuse,'
                    Proving his beauty by succession thine!
                    This were to be new made when thou art old,
                    And see thy blood warm when thou feel'st it cold.""".split()

In [17]:
ngrams = [([test_sentence[i - j - 1] for j in range(CONTEXT_SIZE)],test_sentence[i])
            for i in range(CONTEXT_SIZE, len(test_sentence))]
print(ngrams[:3])

[(['forty', 'When'], 'winters'), (['winters', 'forty'], 'shall'), (['shall', 'winters'], 'besiege')]


In [21]:
vocab = set(test_sentence) # Create vocabulary
word_to_ix = {word: i for i, word in enumerate(vocab)} # Set index for each word

In [22]:
class NGramLanguageModeler(nn.Module):
    def __init__(self, vocab_size, embedding_dim, context_size):
        super(NGramLanguageModeler, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear1 = nn.Linear(context_size * embedding_dim, 128)
        self.linear2 = nn.Linear(128, vocab_size)

    def forward(self, inputs):
        embeds = self.embeddings(inputs).view((1, -1))
        out = F.relu(self.linear1(embeds))
        out = self.linear2(out)
        log_probs = F.log_softmax(out, dim=1)
        return log_probs

In [27]:
losses, loss_function,  = [], nn.NLLLoss()
model = NGramLanguageModeler(len(vocab), EMBEDDING_DIM, CONTEXT_SIZE)
optimizer = optim.SGD(model.parameters(), lr=0.001)

In [29]:
for epoch in range(10):
    total_loss = 0
    for context, target in ngrams:

        # Step 1. Prepare the inputs to be passed to the model (i.e, turn the words into integer indices and wrap them in tensors)
        context_idxs = torch.tensor([word_to_ix[w] for w in context], dtype=torch.long)

        # Step 2. Recall that torch *accumulates* gradients. Before passing in a new instance, you need to zero out the gradients from the old instance
        model.zero_grad()

        # Step 3. Run the forward pass, getting log probabilities over next words
        log_probs = model(context_idxs)

        # Step 4. Compute your loss function. (Again, Torch wants the target word wrapped in a tensor)
        loss = loss_function(log_probs, torch.tensor([word_to_ix[target]], dtype=torch.long))

        # Step 5. Do the backward pass and update the gradient
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    losses.append(total_loss)

print(losses)  # The loss decreased every iteration over the training data!

print(model.embeddings.weight[word_to_ix["beauty"]]) # To get the embedding of a particular word, e.g. "beauty"

[523.5832235813141, 521.1733198165894, 518.7791337966919, 516.4004817008972, 514.0349364280701, 511.6818380355835, 509.34110713005066, 507.01249742507935, 504.695020198822, 502.38773560523987, 500.0894045829773, 497.80142641067505, 495.52174711227417, 493.2497799396515, 490.98467230796814, 488.72505617141724, 486.47084975242615, 484.2217872142792, 481.9772460460663, 479.7362685203552]
tensor([-1.2765,  0.9175, -0.4352, -0.7390, -0.4760, -0.9921, -1.4260, -0.1818,
         0.6982,  0.8430], grad_fn=<SelectBackward0>)
